In [3]:
# Local environment check for QuimicaAutomocio P2.
# The original eChem notebooks are Colab-oriented; this one is designed for local use.
import numpy as np
import py3Dmol
import veloxchem as vlx

required = ["ScfRestrictedDriver", "OptimizationDriver", "VibrationalAnalysis"]
missing = [name for name in required if not hasattr(vlx, name)]
if missing:
    raise RuntimeError(f"VeloxChem is missing required classes: {missing}")

print("VeloxChem", getattr(vlx, "__version__", "unknown"))
print("Environment OK")

VeloxChem 1.0rc4
Environment OK


In [4]:
molecule = vlx.Molecule.read_smiles("CCC/C=C/C=[NH+]\c1ccc(C[C@H](N)C(=O)O)cc1")

print(molecule.get_xyz_string())
molecule.show(atom_indices=True, width=520, height=360)

<>:1: SyntaxWarning: invalid escape sequence '\c'
<>:1: SyntaxWarning: invalid escape sequence '\c'
C:\Users\Nayanika\AppData\Local\Temp\ipykernel_14488\712903270.py:1: SyntaxWarning: invalid escape sequence '\c'
  molecule = vlx.Molecule.read_smiles("CCC/C=C/C=[NH+]\c1ccc(C[C@H](N)C(=O)O)cc1")


40

C             -5.046198000000        -0.433799000000         2.391983000000
C             -4.302762000000        -1.174822000000         1.281105000000
C             -3.587725000000        -2.432318000000         1.797199000000
C             -2.477205000000        -2.106661000000         2.756718000000
C             -1.351872000000        -1.514134000000         2.349668000000
C             -0.273329000000        -1.196589000000         3.309778000000
N              0.882787000000        -0.704474000000         2.957643000000
C              1.323561000000        -0.515067000000         1.602859000000
C              1.331399000000        -1.581881000000         0.688563000000
C              1.799309000000        -1.390550000000        -0.614378000000
C              2.292467000000        -0.137973000000        -1.012983000000
C              2.764679000000         0.081262000000        -2.427527000000
C              1.648359000000         0.676096000000        -3.303962000000
N       

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
basis = vlx.MolecularBasis.read(molecule, "def2-svp")

scf_drv = vlx.ScfRestrictedDriver()
scf_drv.xcfun = "b3lyp"
scf_drv.dispersion = True
scf_drv.ostream.mute()

opt_drv = vlx.OptimizationDriver(scf_drv)
opt_drv.ostream.mute()
opt_drv.max_iter = 100

opt_results = opt_drv.compute(molecule, basis)
opt_molecule = opt_results["final_molecule"]

print("Optimized geometry:")
print(opt_molecule.get_xyz_string())
print("Optimization steps:", len(opt_results.get("opt_energies", [])))
opt_molecule.show(atom_indices=True, width=520, height=360)

In [ ]:
vib_drv = vlx.VibrationalAnalysis(scf_drv)
vib_drv.ostream.mute()
vib_drv.do_ir = True

vib_results = vib_drv.compute(opt_molecule, basis)

frequencies = np.array(vib_results["vib_frequencies"], dtype=float)
normal_modes = np.array(vib_results["normal_modes"], dtype=float)
reduced_masses = np.array(vib_results["reduced_masses"], dtype=float)
force_constants = np.array(vib_results["force_constants"], dtype=float)
ir_intensities = np.array(vib_results.get("ir_intensities", np.zeros_like(frequencies)), dtype=float)

print("mode  frequency / cm^-1   red. mass / amu   force const. / mdyn A^-1   IR / km mol^-1")
for i, freq in enumerate(frequencies, start=1):
    print(f"{i:>4d}  {freq:>16.2f}  {reduced_masses[i-1]:>15.4f}  {force_constants[i-1]:>25.4f}  {ir_intensities[i-1]:>14.2f}")

mode  frequency / cm^-1   red. mass / amu   force const. / mdyn A^-1   IR / km mol^-1
   1            115.39           1.2744                     0.0100            0.93
   2            135.80           2.2111                     0.0240            2.91
   3            251.37           4.0789                     0.1518            0.02
   4            256.72           2.8993                     0.1126            0.82
   5            309.71           3.8926                     0.2200            0.01
   6            444.50           1.8697                     0.2176           38.17
   7            450.30           1.5481                     0.1849           20.47
   8            460.17           3.1155                     0.3887            2.91
   9            499.20           5.4105                     0.7944            0.17
  10            593.13           3.3919                     0.7030            0.38
  11            619.32           6.6031                     1.4922            1.46
 

In [ ]:
def normal_mode_trajectory(molecule, mode, amplitude=0.45, frames=32):
    labels = list(molecule.get_labels())
    coords = np.array(molecule.get_coordinates_in_angstrom(), dtype=float)
    mode = np.array(mode, dtype=float)

    chunks = []
    for phase in np.linspace(0.0, 2.0 * np.pi, frames, endpoint=False):
        displaced = coords + amplitude * np.sin(phase) * mode
        chunks.append(str(len(labels)))
        chunks.append("normal-mode frame")
        for label, xyz in zip(labels, displaced):
            chunks.append(f"{label:2s} {xyz[0]: .8f} {xyz[1]: .8f} {xyz[2]: .8f}")
    return "\n".join(chunks) + "\n"

def show_mode(molecule, mode_index, amplitude=0.45, frames=32):
    traj = normal_mode_trajectory(molecule, normal_modes[mode_index], amplitude=amplitude, frames=frames)
    view = py3Dmol.view(width=650, height=430)
    view.addModelsAsFrames(traj, "xyz")
    view.setStyle({"stick": {"radius": 0.16}, "sphere": {"scale": 0.24}})
    view.animate({"loop": "backAndForth", "reps": 0, "interval": 80})
    view.zoomTo()
    return view

# Python index: 0 is the first vibrational mode. Change this value.
mode_index = 0
print(f"Showing mode {mode_index}: {frequencies[mode_index]:.2f} cm^-1")
show_mode(opt_molecule, mode_index, amplitude=0.45, frames=32).show()
print(normal_modes[mode_index])

Showing mode 0: 115.39 cm^-1


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

[[ 0.0146229   0.04909266  0.02091396]
 [ 0.01195333  0.04018638  0.01712348]
 [ 0.00777821  0.02614307  0.0111558 ]
 [ 0.01031226  0.03460798  0.01474185]
 [ 0.00984283  0.03308745  0.01410165]
 [-0.02510699 -0.08439436 -0.03598546]
 [-0.00603748 -0.02025967 -0.00862008]
 [-0.0156616  -0.05258871 -0.02239175]
 [-0.00747921 -0.02510176 -0.01069828]
 [ 0.00799788  0.02688502  0.01144594]
 [ 0.01560208  0.05226539  0.02234177]
 [ 0.00582006  0.01946883  0.00825971]
 [ 0.10600817  0.36491908  0.15559892]
 [ 0.35274499 -0.4929827  -0.15509007]
 [-0.56419027 -0.22635849 -0.15158412]
 [-0.01214342 -0.04076115 -0.01735448]
 [-0.02936895 -0.09862262 -0.04200493]
 [-0.01420868 -0.04773801 -0.02035169]
 [ 0.01277334  0.04291706  0.01827085]]
